# NYC HVFHV — Data Preprocessing

Applies data quality filters and type conversions to the raw HVFHV trip data 
(see `01_data_understanding.ipynb` for the EDA and rationale behind each step), 
producing a clean table (`fhvhv_clean`) for downstream hypothesis testing.

**Input:** `fhvhv_raw` (materialized from `../data/raw/fhvhv_tripdata_2026-*.parquet`)
**Reference Data:** `../data/reference/taxi_zone_lookup.csv`
**Output:** `fhvhv_clean`

## Preprocessing Steps Applied
- Removed `trip_miles <= 0` (~11.5K rows, 0.01%)
- Removed invalid trip duration — `dropoff_datetime <= pickup_datetime` (2 rows)
- Removed `base_passenger_fare <= 0` (0.14%)
- Removed `driver_pay <= 0` (0.06%)
- Removed timestamp discrepancies — `on_scene_datetime` before `request_datetime` (~1.9M rows, 1.8%) 
  and `on_scene_datetime` after `pickup_datetime` (~3.2K rows, 0.003%)
- Converted Y/N flags to booleans — `shared_request_flag`, `shared_match_flag`, 
  `access_a_ride_flag`, `wav_request_flag`, `wav_match_flag`
- Mapped provider codes to names — HV0003 → Uber, HV0005 → Lyft
- Joined the NYC Taxi Zone Lookup table to add pickup and dropoff zone, borough, and service zone information

In [16]:
import duckdb

con = duckdb.connect("../data/nyc_fhvhv_2026.duckdb")

# Our source of truth is the raw fhvhv data
con.execute("""
CREATE OR REPLACE TABLE fhvhv_raw AS
SELECT DISTINCT *
FROM read_parquet('../data/raw/fhvhv_tripdata_2026-*.parquet');

CREATE OR REPLACE TABLE zone_lookup AS
SELECT * FROM read_csv('../data/reference/taxi_zone_lookup.csv');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [17]:
con.sql("""
SELECT COUNT(*) FROM fhvhv_raw;
""").df()

,count_star()
0,105996113


In [18]:
con.sql("""
SELECT COUNT(*) FROM zone_lookup;
""").df()

,count_star()
0,265


In [22]:
con.sql("""
SELECT 
    * 
FROM zone_lookup
LIMIT 5;
""").df()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [24]:
# Create a cleaned version of the fhvhv data
con.execute("""
CREATE OR REPLACE TABLE fhvhv_clean AS
SELECT
    CASE 
        WHEN hvfhs_license_num = 'HV0003' THEN 'Uber'
        WHEN hvfhs_license_num = 'HV0005' THEN 'Lyft'
        ELSE hvfhs_license_num
    END AS provider_name,

    -- License number
    t.dispatching_base_num,
    t.originating_base_num,

    -- Datetime columns
    t.request_datetime,
    t.on_scene_datetime,
    t.pickup_datetime,
    t.dropoff_datetime,

    -- Pickup location
    t.PULocationID,
    pu.Borough AS pickup_borough,
    pu.Zone AS pickup_zone,
    pu.service_zone AS pickup_service_zone,

    -- Dropoff location
    t.DOLocationID,
    dz.Borough AS dropoff_borough,
    dz.Zone AS dropoff_zone,
    dz.service_zone AS dropoff_service_zone,

    -- Numeric columns
    t.trip_miles,
    t.trip_time,
    t.base_passenger_fare,
    t.tolls,
    t.bcf,
    t.sales_tax,
    t.congestion_surcharge,
    t.airport_fee,
    t.tips,
    t.driver_pay,
    t.cbd_congestion_fee,

    -- Flags
    (t.shared_request_flag = 'Y') AS shared_request_flag,
    (t.shared_match_flag = 'Y')   AS shared_match_flag,
    (t.access_a_ride_flag = 'Y')  AS access_a_ride_flag,
    (t.wav_request_flag = 'Y')    AS wav_request_flag,
    (t.wav_match_flag = 'Y')      AS wav_match_flag

FROM fhvhv_raw t
JOIN zone_lookup pu ON t.PULocationID = pu.LocationID
JOIN zone_lookup dz ON t.DOLocationID = dz.LocationID
WHERE t.trip_miles > 0
    AND t.dropoff_datetime > t.pickup_datetime
    AND t.base_passenger_fare > 0
    AND t.driver_pay > 0
    AND t.on_scene_datetime >= t.request_datetime
    AND t.on_scene_datetime <= t.pickup_datetime;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [25]:
con.sql("""
SELECT
    *
FROM fhvhv_clean
LIMIT 5;
""").df()

,provider_name,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,pickup_borough,pickup_zone,...,congestion_surcharge,airport_fee,tips,driver_pay,cbd_congestion_fee,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,Uber,B03404,B03404,2026-05-27 10:24:51,2026-05-27 10:25:56,2026-05-27 10:26:37,2026-05-27 10:38:01,42,Manhattan,Central Harlem North,...,0.00,0.0,0.0,10.04,0.0,False,False,False,False,False
1,Uber,B03404,B03404,2026-05-27 10:44:39,2026-05-27 10:50:14,2026-05-27 10:50:33,2026-05-27 11:03:42,107,Manhattan,Gramercy,...,2.75,0.0,0.0,10.73,1.5,False,False,False,False,False
2,Lyft,B03406,NaN,2026-05-27 10:47:37,2026-05-27 10:53:19,2026-05-27 10:53:20,2026-05-27 10:57:08,165,Brooklyn,Midwood,...,0.00,0.0,0.0,4.00,0.0,False,False,False,False,True
3,Lyft,B03406,NaN,2026-05-27 10:30:54,2026-05-27 10:32:11,2026-05-27 10:32:15,2026-05-27 10:57:43,141,Manhattan,Lenox Hill West,...,2.75,0.0,0.0,27.88,0.0,False,False,False,False,False
4,Uber,B03404,B03404,2026-05-27 09:49:09,2026-05-27 10:06:18,2026-05-27 10:06:32,2026-05-27 10:57:20,17,Brooklyn,Bedford,...,0.00,0.0,0.0,46.55,0.0,False,False,False,False,False


In [26]:
con.sql("""
DESCRIBE fhvhv_clean;
""").df()

,column_name,column_type,null,key,default,extra
0,provider_name,VARCHAR,YES,None,None,None
1,dispatching_base_num,VARCHAR,YES,None,None,None
2,originating_base_num,VARCHAR,YES,None,None,None
3,request_datetime,TIMESTAMP,YES,None,None,None
4,on_scene_datetime,TIMESTAMP,YES,None,None,None
5,pickup_datetime,TIMESTAMP,YES,None,None,None
6,dropoff_datetime,TIMESTAMP,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,pickup_borough,VARCHAR,YES,None,None,None
9,pickup_zone,VARCHAR,YES,None,None,None


In [27]:
con.sql("""
SELECT COUNT(*) FROM fhvhv_clean;
""").df()

,count_star()
0,103921454


In [28]:
con.sql("""
SELECT
    SUM(trip_miles <= 0) AS invalid_trip_miles,
    SUM(dropoff_datetime <= pickup_datetime) AS invalid_duration,
    SUM(base_passenger_fare <=0) AS invalid_fare,
    SUM(driver_pay <=0) AS invalid_driver_pay,
    SUM(on_scene_datetime < request_datetime) AS invalid_request_time,
    SUM(on_scene_datetime > pickup_datetime) AS invalid_pickup_time
FROM fhvhv_clean;
""").df()

,invalid_trip_miles,invalid_duration,invalid_fare,invalid_driver_pay,invalid_request_time,invalid_pickup_time
0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
con.sql("""
SELECT 
    provider_name,
    COUNT(*) AS total_trips,
    SUM(CASE WHEN originating_base_num IS NULL THEN 1 ELSE 0 END) AS nulls
FROM fhvhv_clean
GROUP BY provider_name;
""").df()

,provider_name,total_trips,nulls
0,Lyft,29310514,29266702.0
1,Uber,74610940,0.0


In [30]:
con.sql("""
SELECT
    COUNT(*) AS total_trips,
    SUM(CASE WHEN base_passenger_fare <= 0 THEN 1 ELSE 0 END) AS invalid_base_passenger_fare_count,
    SUM(CASE WHEN driver_pay <= 0 THEN 1 ELSE 0 END) AS invalid_driver_pay_count,
    (SUM(CASE WHEN base_passenger_fare <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS invalid_base_passenger_fare_pct,
    (SUM(CASE WHEN driver_pay <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS invalid_driver_pay_pct
FROM fhvhv_clean;
""").df()

,total_trips,invalid_base_passenger_fare_count,invalid_driver_pay_count,invalid_base_passenger_fare_pct,invalid_driver_pay_pct
0,103921454,0.0,0.0,0.0,0.0


In [31]:
con.close()